In [1]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb 'timesfm[torch]'

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

Upload your kaggle.json


Saving kaggle.json to kaggle.json
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adola23 (adola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100% 2.70M/2.70M [00:00<00:00, 216MB/s]

2026/07/11 17:13:17 INFO mlflow.store.db.utils: Updating database tables


In [2]:
import numpy as np
import pandas as pd
import torch
import mlflow
import mlflow.sklearn
import wandb
import timesfm
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline

In [3]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
train.shape

(421570, 5)

In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [5]:
# TimesFM: google-is pretrained foundation modeli. treningi ar gvchirdeba,
# yovel seriaze zero-shot prognozs vigebt. modeli pickle-shi ar inaxeba
# (200M+ parametria), predict-is dros huggingface-dan chamoitvirteba.
# bibliotekas ori versiis api aqvs (1.x da 2.5+), orive mxardacherilia
NEW_TIMESFM_API = not hasattr(timesfm, 'TimesFm')

class TimesFMForecaster(BaseEstimator):
    _shared_model = None

    def __init__(self, context_len=128, horizon_len=48, batch_size=64):
        self.context_len = context_len
        self.horizon_len = horizon_len
        self.batch_size = batch_size

    def _model(self):
        cls = TimesFMForecaster
        if cls._shared_model is None:
            if NEW_TIMESFM_API:
                m = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
                m.compile(timesfm.ForecastConfig(
                    max_context=max(self.context_len, 256),
                    max_horizon=max(self.horizon_len, 64),
                    normalize_inputs=True,
                ))
                cls._shared_model = m
            else:
                cls._shared_model = timesfm.TimesFm(
                    hparams=timesfm.TimesFmHparams(
                        backend='gpu' if torch.cuda.is_available() else 'cpu',
                        per_core_batch_size=self.batch_size,
                        context_len=self.context_len,
                        horizon_len=self.horizon_len,
                    ),
                    checkpoint=timesfm.TimesFmCheckpoint(
                        huggingface_repo_id='google/timesfm-2.0-500m-pytorch'),
                )
        return cls._shared_model

    def _forecast(self, contexts):
        model = self._model()
        outs = []
        for i in range(0, len(contexts), self.batch_size):
            chunk = contexts[i:i + self.batch_size]
            if NEW_TIMESFM_API:
                point, _ = model.forecast(horizon=self.horizon_len, inputs=chunk)
            else:
                point, _ = model.forecast(chunk, freq=[1] * len(chunk))
            outs.append(np.asarray(point))
        return np.concatenate(outs, axis=0)

    def fit(self, X, y):
        df = X[['Store', 'Dept', 'Date']].copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['Weekly_Sales'] = y.values if hasattr(y, 'values') else y

        self.history_ = df.pivot_table(index=['Store', 'Dept'], columns='Date',
                                       values='Weekly_Sales').sort_index(axis=1)
        self.series_mean_ = self.history_.mean(axis=1).fillna(0)
        self.last_date_ = self.history_.columns.max()
        return self

    def predict(self, X):
        df = X.copy().reset_index(drop=True)
        df['Date'] = pd.to_datetime(df['Date'])

        contexts = []
        for k in self.history_.index:
            v = self.history_.loc[k].dropna().values[-self.context_len:]
            contexts.append(v if len(v) else np.array([0.0]))

        point = self._forecast(contexts)

        future_dates = pd.date_range(self.last_date_ + pd.Timedelta(weeks=1),
                                     periods=self.horizon_len, freq='W-FRI')
        n = min(point.shape[1], len(future_dates))
        fc = pd.DataFrame(point[:, :n], index=self.history_.index, columns=future_dates[:n])
        fc_long = fc.stack()

        idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept'], df['Date']])
        vals = fc_long.reindex(idx).values.astype(float)
        pair_idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept']])
        fallback = self.series_mean_.reindex(pair_idx).values.astype(float)
        global_mean = float(self.series_mean_.mean())

        preds = np.where(np.isnan(vals), np.where(np.isnan(fallback), global_mean, fallback), vals)
        return np.clip(preds, 0, None)

In [6]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('TimesFM_Training')

<Experiment: artifact_location='/content/walmart-sales-forecasting/mlruns/13', creation_time=1783785287632, effective_trace_archival_retention=None, experiment_id='13', last_update_time=1783785287632, lifecycle_stage='active', name='TimesFM_Training', tags={}, trace_location=None, workspace='default'>

In [7]:
with mlflow.start_run(run_name='TimesFM_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='TimesFM_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    wandb.log({'negative_sales_rows': n_negative})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

negative_sales_rows,▁
negative_sales_rows,1285


In [8]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

# test-formis holdout: oqtombramde vswavlobt, shemdegi noembridan ivlisamde vafasebt.
# diagnostikam achvena rom es fanjara kaggle-is tests bevrad ukot imeorebs vidre CV
HOLDOUT_CUTOFF = pd.Timestamp('2011-10-28')
HOLDOUT_END = pd.Timestamp('2012-07-27')

In [9]:
params = dict(context_len=128, horizon_len=48)

with mlflow.start_run(run_name='TimesFM_CV'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='TimesFM_CV', config=params, reinit='finish_previous')

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        tm = train['Date'] <= train_end
        vm = (train['Date'] > train_end) & (train['Date'] <= val_end)

        model = TimesFMForecaster(**params)
        model.fit(train.loc[tm, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[tm])
        preds = model.predict(train.loc[vm, ['Store', 'Dept', 'Date', 'IsHoliday']])

        score = wmae(y_train[vm].values, preds, train.loc[vm, 'IsHoliday'].values)
        fold_scores.append(score)
        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'fold': i, 'wmae': score})

    hm = train['Date'] <= HOLDOUT_CUTOFF
    hv = (train['Date'] > HOLDOUT_CUTOFF) & (train['Date'] <= HOLDOUT_END)
    hmodel = TimesFMForecaster(**params)
    hmodel.fit(train.loc[hm, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[hm])
    hpreds = hmodel.predict(train.loc[hv, ['Store', 'Dept', 'Date', 'IsHoliday']])
    wmae_holdout = wmae(y_train[hv].values, hpreds, train.loc[hv, 'IsHoliday'].values)

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    mlflow.log_metric('wmae_holdout', wmae_holdout)
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores)),
               'wmae_holdout': wmae_holdout})
    wandb.finish()

fold_scores, wmae_holdout

config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]

fold,▁▅█
wmae,█▆▁
wmae_holdout,▁
wmae_mean,▁
wmae_std,▁
fold,2
wmae,1672.10056
wmae_holdout,2753.91097
wmae_mean,2731.28327
wmae_std,772.13318


([np.float64(3490.838678109731),
  np.float64(3030.9105677411444),
  np.float64(1672.100563104697)],
 np.float64(2753.9109653595133))

In [10]:
with mlflow.start_run(run_name='TimesFM_Final'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='TimesFM_Final', config=params, reinit='finish_previous')

    pipeline = Pipeline([
        ('model', TimesFMForecaster(**params)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    # 200M-parametriani foundation modeli class-cache-shi zis da cloudpickle mas
    # class-tan ertad inaxavda (886MB pkl, git-shi ver aitvirteba). vasuftavebt
    # log-amde rom pipeline patara darchhes (mxolod history). predict-is dros
    # checkpoint xelaxla chamoitvirteba huggingface-dan
    TimesFMForecaster._shared_model = None

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

2026/07/11 17:37:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/11 17:37:22 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


In [11]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_TimesFM.ipynb in Colab"
    !git push

[main e8c32f0] Run model_experiment_TimesFM.ipynb in Colab
 6 files changed, 69 insertions(+)
 create mode 100644 mlruns/13/models/m-a9debc0cf1534759a1c02a8c84eebe74/artifacts/MLmodel
 create mode 100644 mlruns/13/models/m-a9debc0cf1534759a1c02a8c84eebe74/artifacts/conda.yaml
 create mode 100644 mlruns/13/models/m-a9debc0cf1534759a1c02a8c84eebe74/artifacts/model.pkl
 create mode 100644 mlruns/13/models/m-a9debc0cf1534759a1c02a8c84eebe74/artifacts/python_env.yaml
 create mode 100644 mlruns/13/models/m-a9debc0cf1534759a1c02a8c84eebe74/artifacts/requirements.txt
Enumerating objects: 18, done.
Counting objects: 100% (18/18), done.
Delta compression using up to 2 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (13/13), 1.75 MiB | 1.87 MiB/s, done.
Total 13 (delta 3), reused 3 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/giomamaca/walmart-sales-forecasting.git
   6e8fede..e8c32f0  main -> main
